In [1]:
# Install the necessary packages for Postgres connection and GARCH modeling
%pip install -q psycopg2-binary arch

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import psycopg2
import pandas as pd
import numpy as np
from arch import arch_model
import warnings

# Suppress arch convergence warnings for cleaner output
warnings.filterwarnings('ignore')

# Supabase PostgreSQL connection string
DB_URI = ""
print("Libraries imported and DB URI configured.")

Libraries imported and DB URI configured.


In [3]:
def get_all_symbols(db_uri):
    """Fetches a list of all unique stock symbols in the database."""
    try:
        conn = psycopg2.connect(db_uri)
        query = "SELECT DISTINCT symbol FROM stock_ohlcv ORDER BY symbol"
        df_symbols = pd.read_sql_query(query, conn)
        conn.close()
        return df_symbols['symbol'].tolist()
    except Exception as e:
        print(f"Error fetching symbols: {e}")
        return []

def fetch_stock_data(symbol, db_uri):
    """Fetches historical closing prices for a specific stock."""
    try:
        conn = psycopg2.connect(db_uri)
        query = """
            SELECT date, close 
            FROM stock_ohlcv 
            WHERE symbol = %s
            ORDER BY date ASC
        """
        df = pd.read_sql_query(query, conn, params=(symbol,))
        conn.close()
        
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['close'] = pd.to_numeric(df['close'], errors='coerce')
            df = df.dropna()
        return df
    except Exception as e:
        print(f"Database error for {symbol}: {e}")
        return pd.DataFrame()

In [4]:
def calculate_garch_volatility(df, symbol, horizons=[90, 180], min_history=252):
    """Fits GARCH(1,1) on a stock and returns its forecasted volatility."""
    if df.empty:
        return None
        
    # Calculate daily percentage returns (scaled by 100 for optimizer stability)
    returns = df.set_index('date')['close'].pct_change().dropna() * 100
    
    if len(returns) < min_history:
        return None  # Not enough data to fit the model reliably
        
    try:
        # Define and fit the model
        am = arch_model(returns, vol='Garch', p=1, q=1, mean='Constant', dist='Normal')
        res = am.fit(disp='off')
        
        results = {'Symbol': symbol.upper()}
        max_horizon = max(horizons)
        
        # Generate the forecast
        forecasts = res.forecast(horizon=max_horizon)
        predicted_variances = forecasts.variance.iloc[-1]
        
        # Calculate expected volatility for each specific horizon
        for h in horizons:
            cumulative_variance = predicted_variances.iloc[:h].sum()
            expected_volatility = np.sqrt(cumulative_variance) / 100 # Scale back down
            results[f'Expected_Volatility_{h}d'] = round(expected_volatility, 4)
            
        return results
        
    except Exception as e:
        return None

In [6]:
# 1. Get all symbols from the database
all_symbols = get_all_symbols(DB_URI)
print(f"Found {len(all_symbols)} unique symbols in the database.\n")

target_horizons = [90, 180]
all_results = []

# 2. Loop through every symbol and calculate GARCH volatility
for idx, symbol in enumerate(all_symbols, 1):
    print(f"[{idx}/{len(all_symbols)}] Processing {symbol}...")
    
    df_stock = fetch_stock_data(symbol, DB_URI)
    result = calculate_garch_volatility(df_stock, symbol, horizons=target_horizons)
    
    if result:
        all_results.append(result)

# 3. Compile and Rank the Results


Found 30 unique symbols in the database.

[1/30] Processing ATRL...
[2/30] Processing BAHL...
[3/30] Processing BOP...
[4/30] Processing DGKC...
[5/30] Processing EFERT...
[6/30] Processing ENGROH...
[7/30] Processing FCCL...
[8/30] Processing FFC...
[9/30] Processing GAL...
[10/30] Processing GHNI...
[11/30] Processing HBL...
[12/30] Processing HUBC...
[13/30] Processing LUCK...
[14/30] Processing MARI...
[15/30] Processing MCB...
[16/30] Processing MEBL...
[17/30] Processing MLCF...
[18/30] Processing NBP...
[19/30] Processing OGDC...
[20/30] Processing PAEL...
[21/30] Processing POL...
[22/30] Processing PPL...
[23/30] Processing PRL...
[24/30] Processing PSO...
[25/30] Processing SAZEW...
[26/30] Processing SEARL...
[27/30] Processing SNGP...
[28/30] Processing SSGC...
[29/30] Processing SYS...
[30/30] Processing UBL...

FINAL GARCH(1,1) VOLATILITY RANKINGS (Top 20)


,Symbol,Expected_Volatility_90d,Expected_Volatility_180d,Rank_90d,Rank_180d
0,GHNI,0.4598,0.7200,1,1
1,LUCK,0.3986,0.6676,2,2
2,GAL,0.3863,0.5485,3,4
3,SYS,0.3708,0.6518,4,3
4,UBL,0.3623,0.5164,5,7
5,PRL,0.3523,0.5310,6,5
6,SAZEW,0.3466,0.5041,7,8
7,MLCF,0.3459,0.4690,8,9
8,ENGROH,0.3129,0.5255,9,6
9,PAEL,0.3107,0.4318,10,11


In [7]:
if all_results:
    final_df = pd.DataFrame(all_results)
    
    # Rank for 90 Days (Rank 1 = Highest Volatility)
    final_df['Rank_90d'] = final_df['Expected_Volatility_90d'].rank(ascending=False, method='min').astype(int)
    
    # Rank for 180 Days (Rank 1 = Highest Volatility)
    final_df['Rank_180d'] = final_df['Expected_Volatility_180d'].rank(ascending=False, method='min').astype(int)
    
    # Sort the final table by the 90-day rank
    final_df = final_df.sort_values('Rank_90d').reset_index(drop=True)
    
    print("\n" + "="*60)
    print("FINAL GARCH(1,1) VOLATILITY RANKINGS (Top 20)")
    print("="*60)
    # Display the top 20 most volatile stocks
    display(final_df.head(30))
    
    # Optional: Save the full rankings to a CSV
    # final_df.to_csv("all_stocks_volatility_rankings.csv", index=False)
else:
    print("\nFailed to calculate volatility for any stocks (possibly due to insufficient data history).")


FINAL GARCH(1,1) VOLATILITY RANKINGS (Top 20)


,Symbol,Expected_Volatility_90d,Expected_Volatility_180d,Rank_90d,Rank_180d
0,GHNI,0.4598,0.7200,1,1
1,LUCK,0.3986,0.6676,2,2
2,GAL,0.3863,0.5485,3,4
3,SYS,0.3708,0.6518,4,3
4,UBL,0.3623,0.5164,5,7
5,PRL,0.3523,0.5310,6,5
6,SAZEW,0.3466,0.5041,7,8
7,MLCF,0.3459,0.4690,8,9
8,ENGROH,0.3129,0.5255,9,6
9,PAEL,0.3107,0.4318,10,11


In [ ]:
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime

# ==========================================
# UPSERT RESULTS TO POSTGRESQL (SUPABASE)
# ==========================================
DB_URI = ""

# Check if final_df actually exists in memory before trying to push
if 'final_df' in locals() and not final_df.empty:
    
    # 1. Normalize column names to lowercase to avoid case-sensitivity KeyErrors
    # This turns 'Symbol' -> 'symbol', 'Expected_Volatility_90d' -> 'expected_volatility_90d'
    df_to_push = final_df.copy()
    df_to_push.columns = [str(col).lower() for col in df_to_push.columns]
    
    # 2. Fix missing date issue
    if 'date' not in df_to_push.columns:
        print("⚠️ 'date' column not found in memory. Adding today's date automatically...")
        df_to_push['date'] = datetime.today().strftime('%Y-%m-%d')
        
    print(f"Preparing to push {len(df_to_push)} records to the database...")
    
    # 3. Format records into a list of tuples for PostgreSQL bulk insertion
    records_to_insert = []
    for _, row in df_to_push.iterrows():
        records_to_insert.append((
            row['symbol'],
            row['date'],
            row['expected_volatility_90d'],
            row['expected_volatility_180d'],
            row['rank_90d'],
            row['rank_180d']
        ))
        
    # 4. Connect to Database and Execute Upsert
    try:
        conn = psycopg2.connect(DB_URI)
        cursor = conn.cursor()
        
        # Postgres "Upsert" logic using ON CONFLICT (symbol, date)
        upsert_query = '''
            INSERT INTO garch_volatility_predictions 
            (symbol, date, expected_volatility_90d, expected_volatility_180d, rank_90d, rank_180d)
            VALUES %s
            ON CONFLICT (symbol, date) 
            DO UPDATE SET 
                expected_volatility_90d = EXCLUDED.expected_volatility_90d,
                expected_volatility_180d = EXCLUDED.expected_volatility_180d,
                rank_90d = EXCLUDED.rank_90d,
                rank_180d = EXCLUDED.rank_180d
        '''
        
        # execute_values is a massive speedup for bulk inserts in Postgres
        execute_values(cursor, upsert_query, records_to_insert)
        conn.commit()
        
        print(f"✅ Successfully upserted {len(records_to_insert)} records into 'garch_volatility_predictions'.")
        
    except Exception as e:
        print(f"❌ Database error: {e}")
        if 'conn' in locals():
            conn.rollback() # Undo the transaction if it failed
            
    finally:
        # Always close connections to prevent locking the database
        if 'cursor' in locals():
            cursor.close()
        if 'conn' in locals():
            conn.close()

else:
    print("⚠️ Error: 'final_df' does not exist or is empty. Make sure you run the forecasting cell first.")

⚠️ 'date' column not found in memory. Adding today's date automatically...
Preparing to push 30 records to the database...
✅ Successfully upserted 30 records into 'garch_volatility_predictions'.
